# Task 1 — Pet breed classification

Train and compare custom, multi-task and pretrained models on Oxford-IIIT Pet.

In [ ]:
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = Path('data')
OUTPUT = Path('task1_results')
OUTPUT.mkdir(exist_ok=True)
print('Device:', DEVICE)

## Data exploration and preprocessing

Split trainval by breed and keep the official test split for final evaluation.

In [ ]:
plain = datasets.OxfordIIITPet(ROOT, split='trainval', target_types=['category', 'binary-category'], download=True)
labels = np.array([target[0] for _, target in plain])
train_idx, val_idx = train_test_split(np.arange(len(labels)), test_size=0.2, stratify=labels, random_state=SEED)
np.savez(OUTPUT / 'split.npz', train=train_idx, validation=val_idx)
class_names = plain.classes
sizes = np.array([Image.open(path).size for path in plain._images])
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
pd.Series(labels).value_counts().sort_index().plot.bar(ax=axes[0], title='Images per breed')
axes[1].scatter(sizes[:, 0], sizes[:, 1], alpha=0.2, s=8)
axes[1].set(xlabel='Width', ylabel='Height', title='Image sizes')
plt.show()
print('Train:', len(train_idx), 'Validation:', len(val_idx))
print(pd.DataFrame({'breed': class_names, 'train': np.bincount(labels[train_idx], minlength=37), 'validation': np.bincount(labels[val_idx], minlength=37)}).to_string(index=False))

In [ ]:
base_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
aug_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(), transforms.RandomRotation(10), transforms.ToTensor()])
weights_resnet = models.ResNet50_Weights.DEFAULT
weights_eff = models.EfficientNet_B0_Weights.DEFAULT

def make_loaders(train_transform, eval_transform, batch_size=32):
    train_data = datasets.OxfordIIITPet(ROOT, split='trainval', target_types=['category', 'binary-category'], transform=train_transform)
    val_data = datasets.OxfordIIITPet(ROOT, split='trainval', target_types=['category', 'binary-category'], transform=eval_transform)
    split_path = OUTPUT / 'split.npz'
    if not split_path.exists():
        raise FileNotFoundError('Run the data exploration cell to create the train/validation split.')
    with np.load(split_path) as split:
        train_indices = split['train']
        validation_indices = split['validation']
    generator = torch.Generator().manual_seed(SEED)
    return (DataLoader(Subset(train_data, train_indices), batch_size=batch_size, shuffle=True, generator=generator, num_workers=0),
            DataLoader(Subset(val_data, validation_indices), batch_size=batch_size, shuffle=False, num_workers=0))

## Custom models

Add one technique at a time. The residual block is implemented from scratch.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.BatchNorm2d(channels), nn.ReLU(), nn.Conv2d(channels, channels, 3, padding=1), nn.BatchNorm2d(channels))
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.layers(x))

class PetCNN(nn.Module):
    def __init__(self, batch_norm=False, dropout=False, residual=False, multitask=False):
        super().__init__()
        layers = []
        for incoming, outgoing in [(3, 32), (32, 64), (64, 128), (128, 256)]:
            layers.append(nn.Conv2d(incoming, outgoing, 3, padding=1))
            if batch_norm:
                layers.append(nn.BatchNorm2d(outgoing))
            layers.extend([nn.ReLU(), nn.MaxPool2d(2)])
        if residual:
            layers.append(ResidualBlock(256))
        self.features = nn.Sequential(*layers)
        self.embedding = nn.Sequential(nn.Flatten(), nn.Linear(256 * 14 * 14, 512), nn.ReLU(), nn.Dropout(0.5) if dropout else nn.Identity())
        self.breed_head = nn.Linear(512, 37)
        self.species_head = nn.Linear(512, 2) if multitask else None

    def forward(self, x):
        x = self.embedding(self.features(x))
        breed = self.breed_head(x)
        return (breed, self.species_head(x)) if self.species_head is not None else breed

## Training and validation

Log loss, accuracy and macro F1 every epoch. Select checkpoints using validation macro F1.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, species_weight=0):
    training = optimizer is not None
    model.train(training)
    if training and getattr(model, 'frozen_backbone', False):
        model.backbone.eval()
    total_loss, targets, predictions = 0.0, [], []
    with torch.set_grad_enabled(training):
        for images, target in loader:
            images = images.to(DEVICE)
            breed = target[0].to(DEVICE)
            species = target[1].to(DEVICE)
            if training:
                optimizer.zero_grad()
            output = model(images)
            if isinstance(output, tuple):
                logits, species_logits = output
                loss = criterion(logits, breed) + species_weight * criterion(species_logits, species)
            else:
                logits = output
                loss = criterion(logits, breed)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            targets.extend(breed.cpu().numpy())
            predictions.extend(logits.argmax(1).cpu().numpy())
    return {'loss': total_loss / len(targets), 'accuracy': accuracy_score(targets, predictions), 'f1': f1_score(targets, predictions, labels=np.arange(37), average='macro', zero_division=0)}

def fit(model, train_loader, val_loader, optimizer, name, epochs=15, species_weight=0):
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    writer = SummaryWriter(str(OUTPUT / 'runs' / name))
    best_f1 = -1
    history = []
    for epoch in range(epochs):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer, species_weight)
        val_metrics = run_epoch(model, val_loader, criterion, species_weight=species_weight)
        for key in train_metrics:
            writer.add_scalars(key, {'train': train_metrics[key], 'validation': val_metrics[key]}, epoch + 1)
        history.append({'epoch': epoch + 1, **{'train_' + k: v for k, v in train_metrics.items()}, **{'val_' + k: v for k, v in val_metrics.items()}})
        print(name, epoch + 1, 'train', train_metrics, 'validation', val_metrics)
        if val_metrics['f1'] > best_f1:
            best_f1 = val_metrics['f1']
            torch.save(model.state_dict(), OUTPUT / (name + '.pt'))
    writer.close()
    model.load_state_dict(torch.load(OUTPUT / (name + '.pt'), map_location=DEVICE, weights_only=True))
    return model, pd.DataFrame(history)

In [ ]:
experiments = [
    ('baseline', False, False, False, False),
    ('batch_norm', True, False, False, False),
    ('dropout', True, True, False, False),
    ('augmentation', True, True, False, True),
    ('residual', True, True, True, True),
]
results = []
for name, bn, drop, residual, augment in experiments:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    train_loader, val_loader = make_loaders(aug_transform if augment else base_transform, base_transform)
    model = PetCNN(bn, drop, residual).to(DEVICE)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
    model, history = fit(model, train_loader, val_loader, optimizer, name)
    best = history.loc[history.val_f1.idxmax()]
    results.append({'model': name, 'accuracy': best.val_accuracy, 'macro_f1': best.val_f1, 'epoch': int(best.epoch)})
    history.to_csv(OUTPUT / (name + '_history.csv'), index=False)
    del model, optimizer
ablation = pd.DataFrame(results)
ablation['delta_accuracy'] = ablation.accuracy.diff()
ablation['delta_f1'] = ablation.macro_f1.diff()
display(ablation)
best_name = ablation.loc[ablation.macro_f1.idxmax(), 'model']
best_config = next(item for item in experiments if item[0] == best_name)
print('Selected custom model:', best_name)

## Multi-task learning

Compare two weights for the species loss with the single-task model.

In [ ]:
multitask_results = []
for weight in [0.25, 1.0]:
    _, bn, drop, residual, augment = best_config
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    model = PetCNN(bn, drop, residual, multitask=True).to(DEVICE)
    train_loader, val_loader = make_loaders(aug_transform if augment else base_transform, base_transform)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
    name = 'multitask_' + str(weight)
    model, history = fit(model, train_loader, val_loader, optimizer, name, species_weight=weight)
    best = history.loc[history.val_f1.idxmax()]
    multitask_results.append({'model': name, 'weight': weight, 'accuracy': best.val_accuracy, 'macro_f1': best.val_f1})
    del model, optimizer
multitask_table = pd.DataFrame(multitask_results)
display(multitask_table)
print('Single-task:', ablation.loc[ablation.model == best_name, ['accuracy', 'macro_f1']])
best_multi = multitask_table.loc[multitask_table.macro_f1.idxmax(), 'model']

## Transfer learning

Compare two ImageNet backbones and two strategies for ResNet-50.

In [ ]:
class TransferModel(nn.Module):
    def __init__(self, architecture, frozen=False):
        super().__init__()
        if architecture == 'resnet50':
            self.backbone = models.resnet50(weights=weights_resnet)
            size = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        else:
            self.backbone = models.efficientnet_b0(weights=weights_eff)
            size = self.backbone.classifier[1].in_features
            self.backbone.classifier = nn.Identity()
        self.head = nn.Linear(size, 37)
        self.frozen_backbone = frozen
        if frozen:
            for parameter in self.backbone.parameters():
                parameter.requires_grad = False

    def forward(self, x):
        return self.head(self.backbone(x))

transfer_specs = [('resnet_frozen', 'resnet50', True), ('resnet_finetuned', 'resnet50', False), ('efficientnet_finetuned', 'efficientnet', False)]
transfer_results = []
for name, architecture, frozen in transfer_specs:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    model = TransferModel(architecture, frozen).to(DEVICE)
    transform = weights_resnet.transforms() if architecture == 'resnet50' else weights_eff.transforms()
    train_loader, val_loader = make_loaders(transform, transform)
    optimizer = torch.optim.Adam([{'params': model.backbone.parameters(), 'lr': 1e-5}, {'params': model.head.parameters(), 'lr': 1e-4}])
    model, history = fit(model, train_loader, val_loader, optimizer, name)
    best = history.loc[history.val_f1.idxmax()]
    transfer_results.append({'model': name, 'accuracy': best.val_accuracy, 'macro_f1': best.val_f1})
    del model, optimizer
transfer_table = pd.DataFrame(transfer_results)
display(transfer_table)
best_transfer = transfer_table.loc[transfer_table.macro_f1.idxmax(), 'model']

## Final test evaluation

Evaluate the four selected models after validation choices are fixed.

In [ ]:
def load_selected(name):
    if name == 'baseline':
        model, transform = PetCNN(), base_transform
    elif name == best_name:
        _, bn, drop, residual, _ = best_config
        model, transform = PetCNN(bn, drop, residual), base_transform
    elif name == best_multi:
        _, bn, drop, residual, _ = best_config
        model, transform = PetCNN(bn, drop, residual, multitask=True), base_transform
    else:
        _, architecture, frozen = next(s for s in transfer_specs if s[0] == name)
        model = TransferModel(architecture, frozen)
        transform = weights_resnet.transforms() if architecture == 'resnet50' else weights_eff.transforms()
    model.load_state_dict(torch.load(OUTPUT / (name + '.pt'), map_location=DEVICE, weights_only=True))
    return model.to(DEVICE).eval(), transform

def predict_test(model, transform):
    data = datasets.OxfordIIITPet(ROOT, split='test', target_types=['category', 'binary-category'], transform=transform, download=True)
    loader = DataLoader(data, batch_size=32, shuffle=False, num_workers=0)
    targets, scores = [], []
    with torch.no_grad():
        for images, target in loader:
            output = model(images.to(DEVICE))
            logits = output[0] if isinstance(output, tuple) else output
            scores.append(logits.cpu())
            targets.extend(target[0].numpy())
    return np.array(targets), torch.cat(scores).numpy()

selected = [('baseline', 'baseline'), ('custom', best_name), ('multitask', best_multi), ('transfer', best_transfer)]
test_results, test_outputs = [], {}
for label, name in selected:
    model, transform = load_selected(name)
    targets, scores = predict_test(model, transform)
    preds = scores.argmax(axis=1)
    top5 = np.argsort(scores, axis=1)[:, -5:]
    test_results.append({'model': label, 'checkpoint': name, 'accuracy': accuracy_score(targets, preds), 'top5': np.mean([y in row for y, row in zip(targets, top5)]), 'macro_f1': f1_score(targets, preds, labels=np.arange(37), average='macro', zero_division=0)})
    test_outputs[label] = (targets, scores)
    del model
display(pd.DataFrame(test_results))

## Error analysis and Grad-CAM

The best model is selected using validation macro F1. Grad-CAM uses the predicted class.

In [ ]:
candidates = [('baseline', ablation.iloc[0].macro_f1), ('custom', ablation.macro_f1.max()), ('multitask', multitask_table.macro_f1.max()), ('transfer', transfer_table.macro_f1.max())]
best_label = max(candidates, key=lambda item: item[1])[0]
best_checkpoint = dict(selected)[best_label]
targets, scores = test_outputs[best_label]
preds = scores.argmax(axis=1)
cm = confusion_matrix(targets, preds, labels=np.arange(37))
fig, ax = plt.subplots(figsize=(16, 14))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(37), class_names, rotation=90)
ax.set_yticks(range(37), class_names)
ax.set(xlabel='Predicted', ylabel='True', title='Confusion matrix')
plt.show()
pairs = [(cm[i, j] + cm[j, i], class_names[i], class_names[j]) for i in range(37) for j in range(i + 1, 37)]
display(pd.DataFrame(sorted(pairs, reverse=True)[:5], columns=['count', 'breed 1', 'breed 2']))

In [ ]:
model, transform = load_selected(best_checkpoint)
if isinstance(model, PetCNN):
    conv_layer = [layer for layer in model.features.modules() if isinstance(layer, nn.Conv2d)][-1]
else:
    conv_layer = [layer for layer in model.backbone.modules() if isinstance(layer, nn.Conv2d)][-1]
for layer in model.modules():
    if isinstance(layer, nn.ReLU):
        layer.inplace = False
activations, gradients = [], []
def save_activation(module, inputs, output):
    activations.append(output.detach())
def save_gradient(module, grad_input, grad_output):
    gradients.append(grad_output[0].detach())
forward_hook = conv_layer.register_forward_hook(save_activation)
backward_hook = conv_layer.register_full_backward_hook(save_gradient)
raw_test = datasets.OxfordIIITPet(ROOT, split='test', target_types=['category', 'binary-category'])
indices = np.concatenate([np.flatnonzero(preds == targets)[:4], np.flatnonzero(preds != targets)[:4]])
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, idx in zip(axes.flat, indices):
    image, _ = raw_test[int(idx)]
    tensor = transform(image).unsqueeze(0).to(DEVICE)
    activations.clear()
    gradients.clear()
    model.zero_grad()
    output = model(tensor)
    logits = output[0] if isinstance(output, tuple) else output
    logits[0, int(preds[idx])].backward()
    weights = gradients[-1].mean(dim=(2, 3), keepdim=True)
    heatmap = torch.relu((weights * activations[-1]).sum(dim=1, keepdim=True))
    heatmap = nn.functional.interpolate(heatmap, size=tensor.shape[-2:], mode='bilinear', align_corners=False)[0, 0].cpu().numpy()
    heatmap /= max(float(heatmap.max()), 1e-8)
    # Display the same spatially transformed image used by the model.
    shown = tensor[0].detach().cpu()
    if best_label == 'transfer':
        shown = shown * torch.tensor([0.229, 0.224, 0.225])[:, None, None] + torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    ax.imshow(shown.permute(1, 2, 0).clamp(0, 1))
    ax.imshow(heatmap, cmap='jet', alpha=0.4)
    ax.set_title(f'True: {class_names[targets[idx]]}\nPred: {class_names[preds[idx]]}', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()
forward_hook.remove()
backward_hook.remove()

## Conclusion

Compare the validation ablation, multi-task and transfer results. Interpret the confusion pairs and Grad-CAM images after running the experiments, including whether attention falls on the animal or background.